In [1]:
import pandas as pd
import datetime as dt

In [2]:
def replace_timestamps_with_step_times(df):
    step_names = ["prefetch", "fasterq_dump", "star", "deseq2_star"]
    for step_name in step_names:
        start_times = (pd.to_datetime(df[f"{step_name}_start_time"]) - dt.datetime(1970,1,1)).dt.total_seconds()
        end_times = (pd.to_datetime(df[f"{step_name}_end_time"]) - dt.datetime(1970,1,1)).dt.total_seconds()
        df[f"{step_name} [s]"] = end_times - start_times
        df = df.drop(columns=[f"{step_name}_end_time", f"{step_name}_start_time"])

    return df

In [3]:
df_toplevel = pd.read_csv("data/neardata-toplevel-test.csv", index_col=0)
df_primary = pd.read_csv("data/neardata-primary-test.csv", index_col=0)

df_toplevel = replace_timestamps_with_step_times(df_toplevel).sort_values("SRR_id").reset_index(drop=True)
df_primary = replace_timestamps_with_step_times(df_primary).sort_values("SRR_id").reset_index(drop=True)

assert df_primary["SRR_id"].equals(df_toplevel["SRR_id"])

In [4]:
df = pd.merge(left=df_toplevel, right=df_primary, how="inner", on="SRR_id", suffixes=("_toplevel", "_primary"))

In [5]:
df[["SRR_id", "STAR_mapping_rate [%]_toplevel", "STAR_mapping_rate [%]_primary"]]

,SRR_id,STAR_mapping_rate [%]_toplevel,STAR_mapping_rate [%]_primary
0,SRR10387432,92.35,94.58
1,SRR10387440,92.93,94.84
2,SRR10970615,90.39,93.36
3,SRR11226964,65.75,68.73
4,SRR11227203,52.80,54.65
5,SRR11227622,73.50,76.98
6,SRR11227918,70.39,73.97
7,SRR11228417,55.37,57.02
8,SRR11509679,76.82,79.53
9,SRR11780522,92.57,95.11


In [8]:
mr_diff = df["STAR_mapping_rate [%]_toplevel"] - df["STAR_mapping_rate [%]_primary"]
print(f"Mapping rate difference: max={abs(mr_diff).max()}, mean={abs(mr_diff).mean()}")

Mapping rate difference: max=5.699999999999989, mean=2.737142857142857
